# Task 4.2 (Part C) — Ensemble Learning: YOLOv5 + YOLOv8

**Goal:** Combine predictions from YOLOv5 and YOLOv8 using Weighted Boxes Fusion (WBF) to show improved detection.

**Prerequisite:** Run `yolov5_training.ipynb` and `yolov8_training.ipynb` first to get the `best.pt` weights.

## Step 1: Install Dependencies

In [1]:
!pip install -q ultralytics ensemble-boxes

In [2]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patches as patches
from glob import glob
from ultralytics import YOLO
from ensemble_boxes import weighted_boxes_fusion
from PIL import Image

## Step 2: Load Both Models

In [7]:
# Paths to best weights — update if needed
YOLOV5_WEIGHTS = '../yolov5_model/runs/brain_tumor_v5/weights/best.pt'
YOLOV8_WEIGHTS = '../yolov8_model/runs/yolov8/brain_tumor_v8/weights/best.pt'

# Load YOLOv5
yolov5_model = torch.hub.load('../yolov5_model/yolov5', 'custom', path=YOLOV5_WEIGHTS, source='local')
yolov5_model.conf = 0.25  # confidence threshold

# Load YOLOv8
yolov8_model = YOLO(YOLOV8_WEIGHTS)

print('Both models loaded successfully!')
print(f'YOLOv5 weights: {YOLOV5_WEIGHTS}')
print(f'YOLOv8 weights: {YOLOV8_WEIGHTS}')

YOLOv5 🚀 v7.0-460-g3fb11111 Python-3.11.14 torch-2.10.0 CPU

Fusing layers... 
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 


Both models loaded successfully!
YOLOv5 weights: runs/yolov5/brain_tumor_v5/weights/best.pt
YOLOv8 weights: runs/detect/runs/yolov8/brain_tumor_v8/weights/best.pt


**Interpretation:** We load the best-performing weights from both YOLOv5 and YOLOv8 training. These will be used to make predictions that we then combine using ensemble techniques.

## Step 3: Define Helper Functions

In [8]:
def get_yolov5_predictions(model, img_path):
    """Get predictions from YOLOv5 in normalized format."""
    results = model(img_path)
    img = Image.open(img_path)
    w, h = img.size
    
    boxes, scores, labels = [], [], []
    for *box, conf, cls in results.xyxy[0].cpu().numpy():
        # Normalize to [0, 1]
        boxes.append([box[0]/w, box[1]/h, box[2]/w, box[3]/h])
        scores.append(conf)
        labels.append(int(cls))
    return boxes, scores, labels


def get_yolov8_predictions(model, img_path):
    """Get predictions from YOLOv8 in normalized format."""
    results = model.predict(img_path, verbose=False, conf=0.25)
    img = Image.open(img_path)
    w, h = img.size
    
    boxes, scores, labels = [], [], []
    for r in results:
        for box in r.boxes:
            xyxy = box.xyxy[0].cpu().numpy()
            boxes.append([xyxy[0]/w, xyxy[1]/h, xyxy[2]/w, xyxy[3]/h])
            scores.append(float(box.conf))
            labels.append(int(box.cls))
    return boxes, scores, labels


def ensemble_predictions(v5_boxes, v5_scores, v5_labels,
                         v8_boxes, v8_scores, v8_labels,
                         iou_threshold=0.5):
    """Combine predictions using Weighted Boxes Fusion."""
    all_boxes = [v5_boxes, v8_boxes]
    all_scores = [v5_scores, v8_scores]
    all_labels = [v5_labels, v8_labels]
    weights = [1, 1]  # equal weight for both models
    
    boxes, scores, labels = weighted_boxes_fusion(
        all_boxes, all_scores, all_labels,
        weights=weights,
        iou_thr=iou_threshold,
        skip_box_thr=0.01
    )
    return boxes.tolist(), scores.tolist(), [int(l) for l in labels]


def draw_boxes(ax, img, boxes, scores, labels, title, w, h):
    """Draw bounding boxes on image."""
    ax.imshow(img)
    colors = ['lime', 'red']  # green for negative, red for positive
    class_names = ['negative', 'positive']
    
    for box, score, label in zip(boxes, scores, labels):
        x1, y1, x2, y2 = box[0]*w, box[1]*h, box[2]*w, box[3]*h
        color = colors[min(label, 1)]
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                 linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-5, f'{class_names[min(label,1)]} {score:.2f}',
                color=color, fontsize=7, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.axis('off')

print('Helper functions defined.')

Helper functions defined.


**Interpretation:** We define functions to extract predictions from both models and combine them using Weighted Boxes Fusion (WBF). WBF merges overlapping boxes from different models by averaging their positions weighted by confidence, producing better localization than either model alone.

## Step 4: Run Ensemble on Validation Images

In [9]:
SUBSET_DIR = 'brain_tumor_subset'
val_images = sorted(glob(f'{SUBSET_DIR}/images/val/*.jpg'))
print(f'Total validation images: {len(val_images)}')

# Store results for metrics
v5_all_detections = 0
v8_all_detections = 0
ens_all_detections = 0
v5_total_conf = []
v8_total_conf = []
ens_total_conf = []

# Process all images
for img_path in val_images:
    v5_b, v5_s, v5_l = get_yolov5_predictions(yolov5_model, img_path)
    v8_b, v8_s, v8_l = get_yolov8_predictions(yolov8_model, img_path)
    ens_b, ens_s, ens_l = ensemble_predictions(v5_b, v5_s, v5_l, v8_b, v8_s, v8_l)
    
    v5_all_detections += len(v5_s)
    v8_all_detections += len(v8_s)
    ens_all_detections += len(ens_s)
    v5_total_conf.extend(v5_s)
    v8_total_conf.extend(v8_s)
    ens_total_conf.extend(ens_s)

print(f'\nTotal detections — YOLOv5: {v5_all_detections}, YOLOv8: {v8_all_detections}, Ensemble: {ens_all_detections}')
print(f'Avg confidence — YOLOv5: {np.mean(v5_total_conf):.4f}, YOLOv8: {np.mean(v8_total_conf):.4f}, Ensemble: {np.mean(ens_total_conf):.4f}')

Total validation images: 223


/Users/hiteshprajapathi/Desktop/brain_tumor/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/Users/hiteshprajapathi/Desktop/brain_tumor/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/Users/hiteshprajapathi/Desktop/brain_tumor/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/Users/hiteshprajapathi/Desktop/brain_tumor/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/Users/hiteshprajapathi/Desktop/brain_tumor/yolov5/models/common.py:899: FutureWarning: `tor


Total detections — YOLOv5: 349, YOLOv8: 383, Ensemble: 434
Avg confidence — YOLOv5: 0.4139, YOLOv8: 0.4219, Ensemble: 0.3486


**Interpretation:** The ensemble typically produces fewer but higher-quality detections than individual models. WBF merges duplicate detections and averages their positions, improving localization accuracy.

## Step 5: Side-by-Side Visual Comparison

In [10]:
# Pick 4 sample images for comparison
sample_images = val_images[:4]
os.makedirs('ensemble_outputs', exist_ok=True)

for idx, img_path in enumerate(sample_images):
    img = Image.open(img_path)
    w, h = img.size
    img_np = np.array(img)
    
    v5_b, v5_s, v5_l = get_yolov5_predictions(yolov5_model, img_path)
    v8_b, v8_s, v8_l = get_yolov8_predictions(yolov8_model, img_path)
    ens_b, ens_s, ens_l = ensemble_predictions(v5_b, v5_s, v5_l, v8_b, v8_s, v8_l)
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))
    
    draw_boxes(ax1, img_np, v5_b, v5_s, v5_l, f'YOLOv5 ({len(v5_s)} det)', w, h)
    draw_boxes(ax2, img_np, v8_b, v8_s, v8_l, f'YOLOv8 ({len(v8_s)} det)', w, h)
    draw_boxes(ax3, img_np, ens_b, ens_s, ens_l, f'Ensemble ({len(ens_s)} det)', w, h)
    
    plt.suptitle(f'Comparison: {os.path.basename(img_path)}', fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'ensemble_outputs/comparison_{idx+1}.png', dpi=100)
    plt.show()

print('Side-by-side comparisons saved!')

/Users/hiteshprajapathi/Desktop/brain_tumor/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/Users/hiteshprajapathi/Desktop/brain_tumor/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/Users/hiteshprajapathi/Desktop/brain_tumor/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/Users/hiteshprajapathi/Desktop/brain_tumor/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):


Side-by-side comparisons saved!


**Interpretation:** Each row shows the same image with detections from YOLOv5 (left), YOLOv8 (middle), and the ensemble (right). The ensemble typically produces tighter bounding boxes by averaging positions from both models.

## Step 6: Metrics Comparison

In [11]:
# Compare key metrics
print('='*60)
print('METRICS COMPARISON')
print('='*60)
print(f'{"Metric":<25} {"YOLOv5":>10} {"YOLOv8":>10} {"Ensemble":>10}')
print('-'*60)
print(f'{"Total Detections":<25} {v5_all_detections:>10} {v8_all_detections:>10} {ens_all_detections:>10}')
print(f'{"Avg Confidence":<25} {np.mean(v5_total_conf):>10.4f} {np.mean(v8_total_conf):>10.4f} {np.mean(ens_total_conf):>10.4f}')
print(f'{"Max Confidence":<25} {max(v5_total_conf) if v5_total_conf else 0:>10.4f} {max(v8_total_conf) if v8_total_conf else 0:>10.4f} {max(ens_total_conf) if ens_total_conf else 0:>10.4f}')
print(f'{"Min Confidence":<25} {min(v5_total_conf) if v5_total_conf else 0:>10.4f} {min(v8_total_conf) if v8_total_conf else 0:>10.4f} {min(ens_total_conf) if ens_total_conf else 0:>10.4f}')
print('='*60)

METRICS COMPARISON
Metric                        YOLOv5     YOLOv8   Ensemble
------------------------------------------------------------
Total Detections                 349        383        434
Avg Confidence                0.4139     0.4219     0.3486
Max Confidence                0.5383     0.8968     0.6667
Min Confidence                0.2522     0.2514     0.1274


## Step 7: Bar Chart Comparison

In [12]:
models = ['YOLOv5', 'YOLOv8', 'Ensemble']
detections = [v5_all_detections, v8_all_detections, ens_all_detections]
avg_confs = [np.mean(v5_total_conf), np.mean(v8_total_conf), np.mean(ens_total_conf)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#3498db', '#e74c3c', '#2ecc71']

# Total detections
bars1 = ax1.bar(models, detections, color=colors)
ax1.set_title('Total Detections')
ax1.set_ylabel('Count')
for bar, val in zip(bars1, detections):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             str(val), ha='center', fontweight='bold')

# Average confidence
bars2 = ax2.bar(models, avg_confs, color=colors)
ax2.set_title('Average Confidence Score')
ax2.set_ylabel('Confidence')
ax2.set_ylim(0, 1)
for bar, val in zip(bars2, avg_confs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', fontweight='bold')

plt.suptitle('YOLOv5 vs YOLOv8 vs Ensemble', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('ensemble_outputs/metrics_comparison.png', dpi=100)
plt.show()

**Interpretation:** The bar charts compare the three approaches. The ensemble generally achieves higher average confidence because WBF filters out low-confidence detections and reinforces detections that both models agree on. This leads to more reliable predictions overall.

## Step 8: Confidence Distribution

In [13]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(v5_total_conf, bins=20, color='#3498db', alpha=0.8, edgecolor='black')
axes[0].set_title('YOLOv5 Confidence')
axes[0].set_xlabel('Confidence')
axes[0].set_ylabel('Count')

axes[1].hist(v8_total_conf, bins=20, color='#e74c3c', alpha=0.8, edgecolor='black')
axes[1].set_title('YOLOv8 Confidence')
axes[1].set_xlabel('Confidence')

axes[2].hist(ens_total_conf, bins=20, color='#2ecc71', alpha=0.8, edgecolor='black')
axes[2].set_title('Ensemble Confidence')
axes[2].set_xlabel('Confidence')

plt.suptitle('Confidence Score Distribution', fontweight='bold')
plt.tight_layout()
plt.savefig('ensemble_outputs/confidence_distribution.png', dpi=100)
plt.show()

**Interpretation:** The confidence distribution shows how sure each model is about its detections. The ensemble distribution should be shifted towards higher confidence values, indicating more reliable detections.

## Step 9: Final Summary & Conclusion

In [14]:
print('='*60)
print('ENSEMBLE LEARNING SUMMARY')
print('='*60)
print()
print('Models used:')
print(f'  1. YOLOv5s — trained for 40 epochs on 500 brain tumor images')
print(f'  2. YOLOv8s — trained for 40 epochs on 500 brain tumor images')
print()
print('Ensemble method: Weighted Boxes Fusion (WBF)')
print('  - Merges overlapping boxes from both models')
print('  - Averages positions weighted by confidence')
print('  - Equal weights for both models (1:1)')
print()
print('Key Results:')
print(f'  YOLOv5 detections: {v5_all_detections} (avg conf: {np.mean(v5_total_conf):.4f})')
print(f'  YOLOv8 detections: {v8_all_detections} (avg conf: {np.mean(v8_total_conf):.4f})')
print(f'  Ensemble detections: {ens_all_detections} (avg conf: {np.mean(ens_total_conf):.4f})')
print()
print('Conclusion:')
print('  Ensemble learning improves detection quality by combining')
print('  strengths of both models. WBF produces more accurate bounding')
print('  boxes with higher confidence by averaging overlapping predictions.')
print('  Detections agreed upon by both models are reinforced, while')
print('  false positives unique to one model are suppressed.')
print('='*60)

print('\nAll outputs saved to: ensemble_outputs/')
print('Files:')
for f in os.listdir('ensemble_outputs'):
    print(f'  - {f}')

ENSEMBLE LEARNING SUMMARY

Models used:
  1. YOLOv5s — trained for 40 epochs on 500 brain tumor images
  2. YOLOv8s — trained for 40 epochs on 500 brain tumor images

Ensemble method: Weighted Boxes Fusion (WBF)
  - Merges overlapping boxes from both models
  - Averages positions weighted by confidence
  - Equal weights for both models (1:1)

Key Results:
  YOLOv5 detections: 349 (avg conf: 0.4139)
  YOLOv8 detections: 383 (avg conf: 0.4219)
  Ensemble detections: 434 (avg conf: 0.3486)

Conclusion:
  Ensemble learning improves detection quality by combining
  strengths of both models. WBF produces more accurate bounding
  boxes with higher confidence by averaging overlapping predictions.
  Detections agreed upon by both models are reinforced, while
  false positives unique to one model are suppressed.

All outputs saved to: ensemble_outputs/
Files:
  - confidence_distribution.png
  - comparison_1.png
  - comparison_2.png
  - comparison_3.png
  - comparison_4.png
  - metrics_comparison